# Run the Minian pipeline notebook over sessions

Driver for `caban.minian_runner` (`plans/minian_batch_runner_plan.md`). The runner executes
the protected template `notebooks/minian_pipeline_BASELINE.ipynb` itself, per session, in the
`minian-native` kernel; it re-implements nothing (§2). Manual use of the WORKING notebooks is
unchanged.

**How to use it:** run setup and choose once, check the preview, then run. Position is
re-derived from each session's `Miniscope/minian_run/run.json`, so a kernel restart costs
nothing and a finished session is never redone.

**Order of work (§11):** dry run on the shortest never-processed HC session → the G06 gate
(`SELECTION = "gate"`, `KEEP_SCRATCH = True`), compared stage by stage against
`minian_intermediate-ORIG/` → nothing is batched before that passes → then the parent plan's
§8 order (borked TFC, LT, CNO, HC).

In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from caban import minian_runner as mr

pd.set_option("display.max_rows", 500, "display.width", 220)

## Setup

Every value is spelled out here. `SCRATCH_ROOT` takes `minian_intermediate` (linked from the
session folder); it is deleted after a successful run unless `KEEP_SCRATCH` — keep it for a gate
run, which compares against it. `N_WORKERS` is dask's worker count (`MINIAN_NWORKERS`); 6 is the
notebook's own default, at 32 GB each. G10 TFC_cond (26 `.avi`) peaked at 12.7 GB with 6. One
session at a time.

In [ ]:
DATA_ROOTS = ["/Volumes/MINISCOPE/SSTCa2"]        # scanned for <mouse>/<day>/<session>/Miniscope
SCRATCH_ROOT = "/Volumes/FUTROLA/minian_scratch"  # minian_intermediate lives here during a run
KEEP_SCRATCH = False                              # True for gate runs
N_WORKERS = 6                                     # dask workers; 6 = the notebook's default
ATTENDED = True                                   # figures inline, stop on failure; False = log and move on

## Choose

`SELECTION`: `"never_processed"` (stubs are listed and left out — parent plan §2), `"gate"`
(the four G06 sessions of parent plan §6.1), or `"labels"` with `LABELS` in the order to run.
`MICE` and `SESSION_TYPES` narrow it; a session type matches from its start, so `["HC"]` is
HC1–HC4 and `["LT", "CNO"]` is LT1/LT2 plus CNO1/CNO2.

In [ ]:
SELECTION = "never_processed"
LABELS = None                   # e.g. ["G13/2022_01_10-TFC_test_B_1wk/16_35_20-HC2"]
MICE = None                     # e.g. ["G05", "G08"]
SESSION_TYPES = None            # e.g. ["HC"]

items = mr.discover_sessions(roots=DATA_ROOTS)
queue = mr.select_sessions(items, which=SELECTION, mice=MICE, session_types=SESSION_TYPES,
                           labels=LABELS)

## Preview

The queue, one table per mouse, grouped day → session: its type, `.avi` count (1,000 frames each), status
(blank = never run, `DONE`, or `failed` / `interrupted`), and the last run's wall time, peak memory, and the unit count of `S`, `C` and `YrA` as saved (blank until run; a
`YrA` unit set that differs from `C`'s is flagged). `failed` /
`interrupted` sessions stay out of the run until `mr.clear_failed_run(item)` puts them back in
the queue (their run folder is kept as `minian_run-failed-<time>/`), or — when the notebook
itself finished — `mr.resume_after_notebook(item)` completes them without re-running it.

In [ ]:
status = mr.queue_status(queue)
print(status["status"].replace("", "not run").value_counts().to_string())
mr.show_queue(status)

## Run

Serial, one session at a time. Each session: set aside any existing output and link the scratch →
execute the template (the executed notebook is saved into `minian_run/` after every cell, with
progress printed every 10 min) → report → re-encode `minian.mp4` / `minian_mc.mp4` at crf 23 →
delete the scratch → `run.json`. Interrupting stops the notebook's whole process tree and
records the session as `interrupted`.

In [ ]:
records = mr.run_all(queue, attended=ATTENDED, scratch_root=SCRATCH_ROOT,
                     keep_scratch=KEEP_SCRATCH, n_workers=N_WORKERS)
mr.show_queue(mr.queue_status(queue))